In [11]:
import os

import ROOT


def compare(files, objNames, legendTexts, outputPath, suffix):
    colors = [
        # ROOT.kBlack,       
        ROOT.kRed-4,       
        ROOT.kBlue-4, 
        ROOT.kGreen+2,     
        ROOT.kOrange+7,   
        # ROOT.kYellow-7,    
        ROOT.kMagenta-3,   
        ROOT.kCyan-3,      
        ROOT.kSpring-5,    
        ROOT.kViolet-4,    
        ROOT.kTeal-5,    
        ROOT.kGray+1    
    ]
    markerStyles = [
        20,
        20,
        20,
        20,
        20,
        21,
        21,
        21,
        21,
        22,
    ]

    canvas = ROOT.TCanvas("c1", "c1", 1200, 900)
    canvas.SetLeftMargin(0.12)
    canvas.SetBottomMargin(0.15)
    canvas.SetRightMargin(0.10)
    canvas.SetTopMargin(0.08)
    Legend = ROOT.TLegend(0.2, 0.7, 0.6, 0.9)
    Legend.SetBorderSize(0)
    Legend.SetFillStyle(0)
    legendFont = 42
    Legend.SetTextFont(legendFont)  
    Legend.SetTextSize(0.04)

    mins, maxs = [], []
    for i, fpath, objName, color, markerStyle in zip(range(len(files)), files, objNames, colors, markerStyles):
        f = ROOT.TFile.Open(fpath, "read")
        obj = f.Get(objName)
        obj.Scale(0.07)
        mins.append(obj.GetMinimum())
        maxs.append(obj.GetMaximum())
        f.Close()
    yMin = min(mins) * 0.8 if min(mins) > 0 else min(mins) * 1.2
    yMax = max(maxs) * 1.2 if max(maxs) > 0 else max(maxs) * 0.8

    frame = canvas.DrawFrame(0, yMin, 12, yMax)
    frame.GetYaxis().SetTitle("LM Scaling factor")
    frame.GetXaxis().SetTitle("#it{p}_{T} (GeV/#it{c})")

    objs = []
    ratio_objs = []
    for i, fpath, objName, color, markerStyle in zip(range(len(files)), files, objNames, colors, markerStyles):
        f = ROOT.TFile.Open(fpath, "read")
        obj = f.Get(objName)
        obj.Scale(0.07)
        obj.SetDirectory(0) if hasattr(obj, "SetDirectory") else None
        obj.SetLineColor(color)
        obj.SetLineWidth(3)
        obj.SetMarkerColor(color)
        obj.SetMarkerStyle(markerStyle)
        obj.SetMarkerSize(1)
        if files.index(fpath) == 0:
            # obj.SetTitle("Inclusive D^{0} v_{2} in OO collisions")
            # obj.Scale(1/0.07)
            obj.GetYaxis().SetTitle("LM scaling factor")
            # obj.GetYaxis().SetRangeUser(0, 0.5)
            obj.GetXaxis().SetTitle("p_{T} (GeV/c)")
            obj.Draw("same")
        elif files.index(fpath) == len(files)-1:
            obj.SetLineColor(ROOT.kBlack)
            obj.SetLineWidth(3)
            obj.SetMarkerColor(ROOT.kBlack)
            obj.SetMarkerStyle(20)
            obj.SetMarkerSize(1)
            obj.Draw("same")
        else:
            # obj.SetTitle("")
            # obj.GetYaxis().SetTitle("inclusive D^{0} v_{2}")
            # obj.GetXaxis().SetTitle("p_{T} (GeV/c)")
            obj.Draw("same")
        Legend.AddEntry(obj, f"{legendTexts[i]}", "lp")
        objs.append(obj)
        f.Close()
    Legend.Draw()
    # canvas.Draw()
    canvas.Update()
    # the last one is the demestor, and the rest are the numerator, calculate and draw the ratio

    canvas_ratio = ROOT.TCanvas("c2", "c2", 1200, 900)
    canvas_ratio.SetLeftMargin(0.12)
    canvas_ratio.SetBottomMargin(0.15)
    canvas_ratio.SetRightMargin(0.10)
    canvas_ratio.SetTopMargin(0.08)
    frame_ratio = canvas_ratio.DrawFrame(0, 0, 12, 5)
    for i in range(len(objs)-1):
        ratio = objs[i].Clone(f"ratio_{i}")
        ratio.Divide(objs[-1])
        # for b in range(1, ratio.GetNbinsX() + 1):
        #     num = objs[i].GetBinContent(b)
        #     den = objs[-1].GetBinContent(b)
            
        #     if den != 0:
        #         temp_ratio = num / den
        #         ratio.SetBinContent(b, num / den)
        #         # num_err = objs[i].GetBinError(b)
        #         # den_err = objs[-1].GetBinError(b)
        #         # ratio_err = temp_ratio * ((num_err / num) ** 2 + (den_err / den) ** 2) ** 0.5 if num != 0 else 0
        #         # ratio.SetBinError(b, ratio_err)
        #     else:
        #         ratio.SetBinContent(b, 0)
        #         ratio.SetBinError(b, 0)
        ratio.SetLineColor(colors[i])
        ratio.SetLineWidth(3)
        ratio.SetMarkerColor(colors[i])
        ratio.SetMarkerStyle(markerStyles[i])
        ratio.SetMarkerSize(1)
        ratio.GetYaxis().SetTitle("Ratio to Biao sp prompt")
        ratio.GetXaxis().SetTitle("p_{T} (GeV/c)")
        ratio.Draw("same")
        ratio_objs.append(ratio)
    line = ROOT.TLine(0, 1, 12, 1)
    line.SetLineStyle(2)
    line.SetLineColor(ROOT.kBlack)
    line.Draw("same")
    # canvas_ratio.Draw()
    canvas_ratio.Update()

    os.system(f"mkdir -p {outputPath}")
    ouput = outputPath + f"compare_{suffix}.root"
    canvas.SaveAs(outputPath + f"compare_{suffix}.png")
    canvas_ratio.SaveAs(outputPath + f"compare_ratio_{suffix}.png")
    outputfile = ROOT.TFile(ouput, "recreate")
    canvas.Write("c1")
    canvas_ratio.Write("c2")
    for obj in objs:
        obj.Write()
    for obj in ratio_objs:
        obj.Write()
    outputfile.Close()



In [12]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/fitprocedure/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/results_fixed_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/results_free_noBL/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hLMFactor_PtBinAssoc1",
    "hLMFactor_PtBinAssoc1",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc DeltaPhi fixed LM w/o BL",
    "2pc DeltaPhi free LM w/o BL",
    "Biao sp prompt",
]
suffix = "Scaling_AppDeltaPhi"
compare(files, objNames, legendTexts, outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/fitprocedure/compare_Scaling_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/fitprocedure/compare_ratio_Scaling_AppDeltaPhi.png has been created


In [20]:
import os

import ROOT


def compare_mass(files, objNames, legendTexts, outputPath, suffix):
    colors = [
        # ROOT.kBlack,       
        ROOT.kRed-4,       
        ROOT.kBlue-4, 
        ROOT.kGreen+2,     
        ROOT.kOrange+7,   
        # ROOT.kYellow-7,    
        ROOT.kMagenta-3,   
        ROOT.kCyan-3,      
        ROOT.kSpring-5,    
        ROOT.kViolet-4,    
        ROOT.kTeal-5,    
        ROOT.kGray+1    
    ]
    markerStyles = [
        20,
        20,
        20,
        20,
        20,
        21,
        21,
        21,
        21,
        22,
    ]

    canvas = ROOT.TCanvas("c1", "c1", 1200, 900)
    canvas.SetLeftMargin(0.12)
    canvas.SetBottomMargin(0.15)
    canvas.SetRightMargin(0.10)
    canvas.SetTopMargin(0.08)
    Legend = ROOT.TLegend(0.2, 0.7, 0.6, 0.9)
    Legend.SetBorderSize(0)
    Legend.SetFillStyle(0)
    legendFont = 42
    Legend.SetTextFont(legendFont)  
    Legend.SetTextSize(0.04)

    mins, maxs = [], []
    for i, fpath, objName, color, markerStyle in zip(range(len(files)), files, objNames, colors, markerStyles):
        f = ROOT.TFile.Open(fpath, "read")
        obj = f.Get(objName)
        # obj.Scale(0.07)
        mins.append(obj.GetMinimum())
        maxs.append(obj.GetMaximum())
        f.Close()
    yMin = min(mins) * 0.8 if min(mins) > 0 else min(mins) * 1.2
    yMax = max(maxs) * 1.2 if max(maxs) > 0 else max(maxs) * 0.8

    frame = canvas.DrawFrame(0, yMin, 12, yMax)
    frame.GetYaxis().SetTitle("LM Scaling factor")
    frame.GetXaxis().SetTitle("#it{p}_{T} (GeV/#it{c})")

    objs = []
    ratio_objs = []
    for i, fpath, objName, color, markerStyle in zip(range(len(files)), files, objNames, colors, markerStyles):
        f = ROOT.TFile.Open(fpath, "read")
        obj = f.Get(objName)
        # obj.Scale(0.07)
        obj.SetDirectory(0) if hasattr(obj, "SetDirectory") else None
        obj.SetLineColor(color)
        obj.SetLineWidth(3)
        obj.SetMarkerColor(color)
        obj.SetMarkerStyle(markerStyle)
        obj.SetMarkerSize(1)
        if files.index(fpath) == 0:
            # obj.SetTitle("Inclusive D^{0} v_{2} in OO collisions")
            # obj.Scale(1/0.07)
            obj.GetYaxis().SetTitle("LM scaling factor")
            # obj.GetYaxis().SetRangeUser(0, 0.5)
            obj.GetXaxis().SetTitle("p_{T} (GeV/c)")
            obj.Draw("same")
        elif files.index(fpath) == len(files)-1:
            obj.SetLineColor(ROOT.kBlack)
            obj.SetLineWidth(3)
            obj.SetMarkerColor(ROOT.kBlack)
            obj.SetMarkerStyle(20)
            obj.SetMarkerSize(1)
            obj.Draw("same")
        else:
            # obj.SetTitle("")
            # obj.GetYaxis().SetTitle("inclusive D^{0} v_{2}")
            # obj.GetXaxis().SetTitle("p_{T} (GeV/c)")
            obj.Draw("same")
        Legend.AddEntry(obj, f"{legendTexts[i]}", "lp")
        objs.append(obj)
        f.Close()
    Legend.Draw()
    # canvas.Draw()
    canvas.Update()
    # the last one is the demestor, and the rest are the numerator, calculate and draw the ratio

    canvas_ratio = ROOT.TCanvas("c2", "c2", 1200, 900)
    canvas_ratio.SetLeftMargin(0.12)
    canvas_ratio.SetBottomMargin(0.15)
    canvas_ratio.SetRightMargin(0.10)
    canvas_ratio.SetTopMargin(0.08)
    frame_ratio = canvas_ratio.DrawFrame(0, 0, 12, 5)
    for i in range(len(objs)-1):
        ratio = objs[i].Clone(f"ratio_{i}")
        ratio.Divide(objs[-1])
        # for b in range(1, ratio.GetNbinsX() + 1):
        #     num = objs[i].GetBinContent(b)
        #     den = objs[-1].GetBinContent(b)
            
        #     if den != 0:
        #         temp_ratio = num / den
        #         ratio.SetBinContent(b, num / den)
        #         # num_err = objs[i].GetBinError(b)
        #         # den_err = objs[-1].GetBinError(b)
        #         # ratio_err = temp_ratio * ((num_err / num) ** 2 + (den_err / den) ** 2) ** 0.5 if num != 0 else 0
        #         # ratio.SetBinError(b, ratio_err)
        #     else:
        #         ratio.SetBinContent(b, 0)
        #         ratio.SetBinError(b, 0)
        ratio.SetLineColor(colors[i])
        ratio.SetLineWidth(3)
        ratio.SetMarkerColor(colors[i])
        ratio.SetMarkerStyle(markerStyles[i])
        ratio.SetMarkerSize(1)
        ratio.GetYaxis().SetTitle("Ratio to Biao sp prompt")
        ratio.GetXaxis().SetTitle("p_{T} (GeV/c)")
        ratio.Draw("same")
        ratio_objs.append(ratio)
    line = ROOT.TLine(0, 1, 12, 1)
    line.SetLineStyle(2)
    line.SetLineColor(ROOT.kBlack)
    line.Draw("same")
    # canvas_ratio.Draw()
    canvas_ratio.Update()

    os.system(f"mkdir -p {outputPath}")
    ouput = outputPath + f"compare_{suffix}.root"
    canvas.SaveAs(outputPath + f"compare_{suffix}.png")
    canvas_ratio.SaveAs(outputPath + f"compare_ratio_{suffix}.png")
    outputfile = ROOT.TFile(ouput, "recreate")
    canvas.Write("c1")
    canvas_ratio.Write("c2")
    for obj in objs:
        obj.Write()
    for obj in ratio_objs:
        obj.Write()
    outputfile.Close()



In [21]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/fitprocedure/"
files = [
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_wo_NF_sub/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_fixed_woBL/CorrelationFitResults/Output_CorrelationFitting_Root/CorrPhiD0_FinalPlots.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_free_woBL/CorrelationFitResults/Output_CorrelationFitting_Root/CorrPhiD0_FinalPlots.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    # "hVnSimFit",
    "hLMFactor_PtBinAssoc1",
    "hLMFactor_PtBinAssoc1",
    "hV2VsPtPrompt",
]
legendTexts = [
    # "2pc mass w/o NF sub.",
    "2pc mass fixed LM w/o BL",
    "2pc mass free LM w/o BL first mass bin",
    "Biao sp prompt",
]
suffix = "Scaling_Appmass"
compare_mass(files, objNames, legendTexts, outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/fitprocedure/compare_Scaling_Appmass.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/fitprocedure/compare_ratio_Scaling_Appmass.png has been created
